# CogMem Phase 2b — GRPO Training on Middle-Q Episodes

**Q-value triage consolidation:** The core CogMem novelty.
- High Q (>= 0.7): Already trained via SFT/DoRA (Phase 2)
- **Middle Q (0.3-0.7): GRPO RL practice (THIS NOTEBOOK)**
- Low Q (< 0.3): Keep as episodic memory

GRPO generates multiple solutions per problem, executes them against tests,
and learns from which ones pass. The model gets smarter by *practicing*.

**Requires:**
- `memory_bank_bigcode.json` from Phase 1
- `bigcodebench_tasks.jsonl` from Phase 1 (for test cases)
- SFT DoRA adapter from Phase 2 (for merging)

**Flow:**
- Cell 1: Install deps (restart kernel after)
- Cell 2: GPU check
- Cell 3: Load data + triage episodes by Q-value
- Cell 4: GRPO training (~45-60 min on A4000)
- Cell 5: Merge SFT + GRPO adapters
- Cell 6: **Restart kernel!** Start Ollama + create models
- Cell 7: Evaluate base vs SFT-only vs merged

In [ ]:
# Cell 1: Install deps (restart kernel after)
!pip install "transformers==4.43.4" "peft==0.13.2" "accelerate==0.33.0" "bitsandbytes==0.43.3" "datasets==2.20.0" "huggingface-hub>=0.24" "pydantic>=2.0" "trl>=0.9.0" pyyaml -q
!python3 -c "import torch; print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"
!python3 -c "from trl import GRPOTrainer; print('TRL GRPOTrainer OK')"
print("Restart kernel, then run Cell 2")

In [ ]:
# Cell 2: GPU check + clone CogMem
import torch
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")
    if vram < 14:
        print("WARNING: GRPO needs ~12-14GB VRAM. May be tight.")

# Clone/update CogMem
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || (cd /notebooks/CogMem && git pull && git checkout feat/bigcodebench-integration)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

# Kill Ollama to free VRAM for training
import subprocess
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

In [ ]:
# Cell 3: Load memory bank + original tasks, triage by Q-value
import json
from pathlib import Path
from collections import Counter

# Load memory bank (episodes with Q-values from Phase 1)
MB_PATH = "/notebooks/CogMem/results/memory_bank_bigcode.json"
if not Path(MB_PATH).exists():
    # Try converting from checkpoint
    alt = "/notebooks/bigcode_episodes.jsonl"
    if Path(alt).exists():
        episodes = []
        with open(alt) as f:
            for line in f:
                if line.strip():
                    episodes.append(json.loads(line))
        with open(MB_PATH, "w") as f:
            json.dump(episodes, f, indent=2)
        print(f"Converted: {len(episodes)} episodes")
    else:
        raise FileNotFoundError(f"Need {MB_PATH} from Phase 1")

with open(MB_PATH) as f:
    episodes = json.load(f)
print(f"Memory bank: {len(episodes)} episodes")

# Load original BigCodeBench tasks (have test cases)
TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
tasks_by_id = {}
with open(TASKS_PATH) as f:
    for line in f:
        t = json.loads(line)
        tasks_by_id[t["task_id"]] = t
print(f"Tasks loaded: {len(tasks_by_id)}")

# Enrich episodes with test cases from original tasks
enriched = 0
for ep in episodes:
    task = tasks_by_id.get(ep.get("task_id", ""))
    if task:
        ep["test"] = task.get("test", "")
        ep["complete_prompt"] = task.get("complete_prompt", "")
        ep["entry_point"] = task.get("entry_point", "")
        enriched += 1
print(f"Enriched with test cases: {enriched}/{len(episodes)}")

# Q-value triage
Q_HIGH = 0.7
Q_MID = 0.3

high = [ep for ep in episodes if ep.get("q_value", 0) >= Q_HIGH]
middle = [ep for ep in episodes if Q_MID <= ep.get("q_value", 0) < Q_HIGH]
low = [ep for ep in episodes if ep.get("q_value", 0) < Q_MID]

print(f"\nQ-value triage:")
print(f"  High   (Q >= {Q_HIGH}): {len(high)} episodes -> already SFT'd")
print(f"  Middle ({Q_MID} <= Q < {Q_HIGH}): {len(middle)} episodes -> GRPO")
print(f"  Low    (Q < {Q_MID}): {len(low)} episodes -> episodic only")

# Q-value distribution
q_vals = [ep.get("q_value", 0) for ep in episodes]
print(f"\nQ-value stats: min={min(q_vals):.2f}, max={max(q_vals):.2f}, "
      f"mean={sum(q_vals)/len(q_vals):.2f}")

# If not enough middle-Q episodes, widen the range
if len(middle) < 5:
    print(f"\nWARNING: Only {len(middle)} middle-Q episodes.")
    print("Widening range: using Q < 0.7 (all non-high) for GRPO")
    middle = [ep for ep in episodes if ep.get("q_value", 0) < Q_HIGH]
    # Filter to those with test cases
    middle = [ep for ep in middle if ep.get("test")]
    print(f"  Expanded middle: {len(middle)} episodes (with test cases)")

# Select anchors from high-Q for stability
import random
random.seed(42)
anchor_count = min(10, len(high))
anchors = sorted(high, key=lambda x: x.get("q_value", 0), reverse=True)[:anchor_count]
print(f"  Anchors (high-Q): {len(anchors)} episodes")

In [ ]:
# Cell 4: GRPO Training — manual implementation
# Uses code execution as reward signal. The model generates G solutions
# per problem, executes them, and learns from the reward gradient.
#
# Manual GRPO is more reliable than TRL's GRPOTrainer on A4000
# because we control memory usage precisely.

import gc, os, re, time, json, subprocess, sys, tempfile
import torch
from pathlib import Path
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.environ["TRANSFORMERS_NO_FLASH_ATTENTION"] = "1"

# --- Config ---
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
GRPO_ADAPTER_DIR = "/notebooks/CogMem/adapters/qwen_bigcode_grpo"
G = 4                  # group size: 4 candidates per problem
MAX_STEPS = 200        # stop after 200 steps (or fewer if data runs out)
LR = 5e-6              # low LR for RL stability
BETA = 0.1             # KL penalty
MAX_NEW_TOKENS = 512   # shorter than SFT to save VRAM during generation
TEMPERATURE = 0.8      # diverse generations
EVAL_TIMEOUT = 30      # seconds per test execution

Path(GRPO_ADAPTER_DIR).mkdir(parents=True, exist_ok=True)

# --- Load model (8-bit + DoRA) ---
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map={"":0},
    torch_dtype=torch.float16, attn_implementation="eager",
)
model = prepare_model_for_kbit_training(model)

dora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    use_dora=True,
)

gc.collect(); torch.cuda.empty_cache()
model = get_peft_model(model, dora_config)
model.print_trainable_parameters()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# --- Reward function: execute code against BigCodeBench tests ---
def extract_python_code(text):
    match = re.search(r'```python\s*\n(.*?)```', text, re.DOTALL)
    if match: return match.group(1).strip()
    match = re.search(r'```\s*\n(.*?)```', text, re.DOTALL)
    if match: return match.group(1).strip()
    stripped = text.strip()
    if stripped.startswith(('import ', 'from ', 'def ', 'class ')):
        return stripped
    return stripped

def execute_and_score(code, task_meta):
    """Run code against unittest, return fraction of tests passed."""
    if not code or len(code.strip()) < 10:
        return -0.5
    
    complete_prompt = task_meta.get("complete_prompt", "")
    test_code = task_meta.get("test", "")
    if not test_code:
        return 0.0  # no tests available
    
    script = f"""{complete_prompt}
{code}

{test_code}

import unittest, sys
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestCases)
runner = unittest.TextTestRunner(stream=sys.stderr, verbosity=0)
result = runner.run(suite)
passed = result.testsRun - len(result.failures) - len(result.errors)
total = result.testsRun
print(f"GRPO_REWARD:{{passed}}/{{total}}")
sys.exit(0 if result.wasSuccessful() else 1)
"""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False, encoding='utf-8') as f:
        f.write(script); script_path = f.name
    try:
        result = subprocess.run(
            [sys.executable, script_path],
            capture_output=True, text=True, timeout=EVAL_TIMEOUT,
        )
        output = result.stdout + result.stderr
        match = re.search(r'GRPO_REWARD:(\d+)/(\d+)', output)
        if match:
            p, t = int(match.group(1)), int(match.group(2))
            return p / t if t > 0 else -0.3
        if 'ALL_TESTS_PASSED' in output:
            return 1.0
        return -0.3
    except subprocess.TimeoutExpired:
        return -0.2
    except Exception:
        return -0.5
    finally:
        Path(script_path).unlink(missing_ok=True)

# --- Build GRPO prompts from middle-Q + anchor episodes ---
grpo_data = []
for ep in middle + anchors:
    instruction = ep.get("task_description", "")
    if not instruction or not ep.get("test"):
        continue
    grpo_data.append({
        "prompt": (
            "You are an expert Python programmer. "
            "Write a complete function that solves the following task. "
            "Include all necessary imports. "
            "Put your code in a ```python code block.\n\n"
            f"Task: {instruction}\n"
        ),
        "test": ep["test"],
        "complete_prompt": ep.get("complete_prompt", ""),
        "entry_point": ep.get("entry_point", ""),
        "task_id": ep.get("task_id", ""),
        "zone": "anchor" if ep in anchors else "middle",
    })

print(f"GRPO training data: {len(grpo_data)} problems")
print(f"  (middle-Q: {sum(1 for d in grpo_data if d['zone']=='middle')}, "
      f"anchors: {sum(1 for d in grpo_data if d['zone']=='anchor')})")

if len(grpo_data) < 3:
    raise ValueError("Not enough problems for GRPO. Need >= 3 with test cases.")

# --- GRPO Training Loop ---
print(f"\nStarting GRPO training...")
print(f"  Group size G={G}, max_steps={MAX_STEPS}, lr={LR}, beta={BETA}")
print(f"  VRAM free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")
print("=" * 60)

all_rewards = []
model.train()
start_time = time.time()

for step in range(min(MAX_STEPS, len(grpo_data))):
    row = grpo_data[step % len(grpo_data)]
    prompt = row["prompt"]
    task_meta = {"test": row["test"], "complete_prompt": row["complete_prompt"],
                 "entry_point": row["entry_point"]}
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=512).to(model.device)
    
    # 1. Generate G completions
    completions = []
    with torch.no_grad():
        for _ in range(G):
            outputs = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE, do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
            gen_ids = outputs[0][inputs.input_ids.shape[1]:]
            text = tokenizer.decode(gen_ids, skip_special_tokens=True)
            completions.append(text)
    
    # 2. Score each completion by executing against tests
    rewards = []
    for comp in completions:
        code = extract_python_code(comp)
        r = execute_and_score(code, task_meta)
        rewards.append(r)
    
    rewards_t = torch.tensor(rewards, dtype=torch.float32)
    all_rewards.extend(rewards)
    
    # 3. Group-relative advantage
    mean_r = rewards_t.mean()
    std_r = rewards_t.std() + 1e-8
    advantages = (rewards_t - mean_r) / std_r
    
    # 4. GRPO loss: weight log-prob by advantage
    optimizer.zero_grad()
    total_loss = torch.tensor(0.0, device=model.device, requires_grad=True)
    n_updates = 0
    
    for comp, adv in zip(completions, advantages):
        if abs(adv.item()) < 0.01:
            continue  # skip near-zero advantage
        
        full_text = prompt + comp
        comp_ids = tokenizer(
            full_text, return_tensors="pt", truncation=True,
            max_length=MAX_NEW_TOKENS + 512,
        ).to(model.device)
        
        out = model(**comp_ids, labels=comp_ids["input_ids"])
        nll = out.loss
        
        # GRPO: -advantage * log_prob (positive advantage -> increase prob)
        loss = -adv.to(model.device) * (-nll)
        total_loss = total_loss + loss
        n_updates += 1
    
    if n_updates > 0:
        avg_loss = total_loss / n_updates
        final_loss = avg_loss + BETA * avg_loss.abs()
        final_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
    
    # 5. Logging
    elapsed = time.time() - start_time
    if (step + 1) % 5 == 0 or step < 3:
        recent = all_rewards[-G*5:] if len(all_rewards) >= G*5 else all_rewards
        avg_r = sum(recent) / len(recent)
        rate = (step + 1) / elapsed * 3600
        eta = (min(MAX_STEPS, len(grpo_data)) - step - 1) / (rate/3600) if rate > 0 else 0
        print(f"  Step {step+1}/{min(MAX_STEPS, len(grpo_data))}: "
              f"mean_reward={avg_r:.3f}, group=[{', '.join(f'{r:.2f}' for r in rewards)}], "
              f"rate={rate:.0f}/hr, ETA={eta/60:.0f}m")
    
    # Checkpoint every 50 steps
    if (step + 1) % 50 == 0:
        model.save_pretrained(f"{GRPO_ADAPTER_DIR}/checkpoint-{step+1}")
        print(f"  Checkpoint saved at step {step+1}")

# Save final adapter
model.save_pretrained(GRPO_ADAPTER_DIR)
tokenizer.save_pretrained(GRPO_ADAPTER_DIR)

# Report
early = all_rewards[:G*5] if len(all_rewards) >= G*5 else all_rewards
late = all_rewards[-G*5:] if len(all_rewards) >= G*5 else all_rewards
print(f"\n{'='*60}")
print(f"GRPO training complete!")
print(f"  Steps: {min(MAX_STEPS, len(grpo_data))}")
print(f"  Reward: early={sum(early)/len(early):.3f} -> late={sum(late)/len(late):.3f}")
print(f"  Adapter saved to {GRPO_ADAPTER_DIR}")
print(f"  Time: {(time.time()-start_time)/60:.0f} min")

# Save training log
with open(f"{GRPO_ADAPTER_DIR}/grpo_log.json", "w") as f:
    json.dump({
        "rewards": all_rewards,
        "config": {"G": G, "lr": LR, "beta": BETA, "max_steps": MAX_STEPS},
        "early_reward": sum(early)/len(early),
        "late_reward": sum(late)/len(late),
    }, f, indent=2)

del model, optimizer
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 5: Merge SFT + GRPO adapters
# Weighted average: 60% SFT (memorization) + 40% GRPO (reasoning)

import torch, json, os, shutil
from pathlib import Path

SFT_DIR = "/notebooks/CogMem/adapters/qwen_bigcode_dora"
GRPO_DIR = "/notebooks/CogMem/adapters/qwen_bigcode_grpo"
MERGED_DIR = "/notebooks/CogMem/adapters/qwen_bigcode_merged"
SFT_WEIGHT = 0.6
GRPO_WEIGHT = 0.4

Path(MERGED_DIR).mkdir(parents=True, exist_ok=True)

def load_adapter(path):
    sf = os.path.join(path, "adapter_model.safetensors")
    bn = os.path.join(path, "adapter_model.bin")
    if os.path.exists(sf):
        from safetensors.torch import load_file
        return load_file(sf, device="cpu")
    return torch.load(bn, map_location="cpu")

print("Loading SFT adapter...")
sft_state = load_adapter(SFT_DIR)
print(f"  SFT params: {len(sft_state)}")

print("Loading GRPO adapter...")
grpo_state = load_adapter(GRPO_DIR)
print(f"  GRPO params: {len(grpo_state)}")

# Weighted average
merged = {}
shared = 0
for key in set(sft_state.keys()) | set(grpo_state.keys()):
    if key in sft_state and key in grpo_state:
        merged[key] = SFT_WEIGHT * sft_state[key] + GRPO_WEIGHT * grpo_state[key]
        shared += 1
    elif key in sft_state:
        merged[key] = sft_state[key]
    else:
        merged[key] = grpo_state[key]

print(f"  Merged: {shared} shared params")

# Save merged adapter
try:
    from safetensors.torch import save_file
    save_file(merged, os.path.join(MERGED_DIR, "adapter_model.safetensors"))
except ImportError:
    torch.save(merged, os.path.join(MERGED_DIR, "adapter_model.bin"))

# Copy adapter config + tokenizer from SFT
for fname in os.listdir(SFT_DIR):
    if fname in ("adapter_config.json", "tokenizer.json", "tokenizer_config.json",
                 "special_tokens_map.json", "tokenizer.model", "vocab.json", "merges.txt"):
        src = os.path.join(SFT_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(MERGED_DIR, fname))

print(f"Merged adapter saved to {MERGED_DIR}")
print(f"Weights: SFT={SFT_WEIGHT}, GRPO={GRPO_WEIGHT}")

# Now merge adapter into full model for Ollama
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

FULL_MERGED_DIR = "/notebooks/cogmem_qwen_merged_sft_grpo"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

print("\nMerging into full model for Ollama...")
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cpu")
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = PeftModel.from_pretrained(base, MERGED_DIR)
model = model.merge_and_unload()
model.save_pretrained(FULL_MERGED_DIR)
tok.save_pretrained(FULL_MERGED_DIR)
print(f"Full merged model saved to {FULL_MERGED_DIR}")

del model, base
import gc; gc.collect()

In [ ]:
# Cell 6: Restart kernel first! Then start Ollama + create all 3 models
import subprocess, time, os

subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    ["ollama", "serve"],
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)
!curl -sf http://localhost:11434/api/tags > /dev/null && echo "Ollama OK" || echo "ERROR"

# 1. Base model
!ollama pull qwen2.5:3b

QWEN_TEMPLATE = """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""

# 2. SFT-only model
SFT_MERGED = "/notebooks/cogmem_qwen_bigcode_merged"  # from Phase 2
if os.path.exists(SFT_MERGED):
    with open("/notebooks/Modelfile.sft", "w") as f:
        f.write(f'FROM {SFT_MERGED}\n')
        f.write(f'TEMPLATE """{QWEN_TEMPLATE}"""\n')
        f.write('PARAMETER stop <|im_end|>\n')
        f.write('PARAMETER temperature 0\n')
    !ollama create cogmem-sft -f /notebooks/Modelfile.sft
    print("Created cogmem-sft")
else:
    print(f"WARNING: SFT merged model not found at {SFT_MERGED}")

# 3. SFT+GRPO merged model
FULL_MERGED = "/notebooks/cogmem_qwen_merged_sft_grpo"
with open("/notebooks/Modelfile.merged", "w") as f:
    f.write(f'FROM {FULL_MERGED}\n')
    f.write(f'TEMPLATE """{QWEN_TEMPLATE}"""\n')
    f.write('PARAMETER stop <|im_end|>\n')
    f.write('PARAMETER temperature 0\n')
!ollama create cogmem-merged -f /notebooks/Modelfile.merged

!ollama list

# Smoke test
from openai import OpenAI
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
resp = client.chat.completions.create(
    model="cogmem-merged",
    messages=[{"role": "user", "content": "Write a Python function that adds two numbers."}],
    max_tokens=200, temperature=0,
)
print(f"\nSmoke test:\n{resp.choices[0].message.content[:300]}")
print("\nAll models ready! Run Cell 7.")

In [ ]:
# Cell 7: Evaluate base vs SFT-only vs SFT+GRPO merged
# This is the key comparison: does GRPO add value on top of SFT?

import json, sys
from pathlib import Path
from openai import OpenAI

if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

from cogmem.benchmarks.bigcodebench.prompts import format_messages, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Load tasks
tasks = []
with open("/notebooks/bigcodebench_tasks.jsonl") as f:
    for line in f:
        tasks.append(json.loads(line.strip()))

EVAL_SIZE = 100
eval_tasks = tasks[:EVAL_SIZE]

# Models to compare
models = ["qwen2.5:3b", "cogmem-sft", "cogmem-merged"]

results = {}
for model_name in models:
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    passed = 0
    total = 0
    checkpoint = f"/notebooks/eval_grpo_{model_name.replace(':', '_').replace('-', '_')}.jsonl"
    
    # Resume
    done_ids = set()
    if Path(checkpoint).exists():
        with open(checkpoint) as f:
            for line in f:
                if line.strip():
                    ep = json.loads(line)
                    done_ids.add(ep["task_id"])
                    total += 1
                    if ep["passed"]: passed += 1
        print(f"Resumed: {len(done_ids)} done")
    
    remaining = [t for t in eval_tasks if t["task_id"] not in done_ids]
    
    for i, task in enumerate(remaining):
        try:
            messages = format_messages(task, use_instruct=True)
            resp = client.chat.completions.create(
                model=model_name, messages=messages,
                max_tokens=2048, temperature=0,
            )
            response = resp.choices[0].message.content
            code = extract_code(response)
            result = evaluate_solution(task, code, timeout=30, mode="subprocess")
            task_passed = result["passed"]
        except Exception as e:
            task_passed = False
            if total < 3:
                print(f"  ERROR: {e}")
        
        total += 1
        if task_passed: passed += 1
        
        with open(checkpoint, "a") as f:
            f.write(json.dumps({"task_id": task["task_id"], "passed": task_passed}) + "\n")
        
        if (i + 1) % 20 == 0:
            print(f"  [{total}/{EVAL_SIZE}] Pass: {passed}/{total} ({passed/total:.1%})")
    
    results[model_name] = {"passed": passed, "total": total,
                           "rate": passed / total if total > 0 else 0}

# Print comparison
print(f"\n{'='*60}")
print(f"{'RESULTS':^60}")
print(f"{'='*60}")
print(f"{'Model':<25} {'Passed':>8} {'Total':>8} {'Rate':>10}")
print(f"{'-'*55}")
base_rate = 0
for name, r in results.items():
    delta = ""
    if name == "qwen2.5:3b":
        base_rate = r["rate"]
    else:
        delta = f"  (+{r['rate']-base_rate:.1%})"
    print(f"{name:<25} {r['passed']:>8} {r['total']:>8} {r['rate']:>9.1%}{delta}")

if "cogmem-sft" in results and "cogmem-merged" in results:
    sft_r = results["cogmem-sft"]["rate"]
    merged_r = results["cogmem-merged"]["rate"]
    grpo_gain = merged_r - sft_r
    print(f"\nGRPO added: {grpo_gain:+.1%} on top of SFT")
    if grpo_gain > 0:
        print("GRPO training improved the model!")
    elif grpo_gain == 0:
        print("GRPO had no effect — may need more steps or different hyperparams.")
    else:
        print("GRPO hurt — try lower LR or higher beta.")

with open("/notebooks/grpo_eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved to /notebooks/grpo_eval_results.json")

In [ ]:
# Cell 8: Package everything
!tar czf /notebooks/cogmem_grpo_results.tar.gz \
    -C /notebooks/CogMem adapters/qwen_bigcode_grpo/ \
    -C /notebooks/CogMem adapters/qwen_bigcode_merged/ \
    -C /notebooks grpo_eval_results.json
!ls -lh /notebooks/cogmem_grpo_results.tar.gz
print("Download cogmem_grpo_results.tar.gz for next cycle")